In [1]:
import segmentation_models_pytorch as smp
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from albumentations import (RandomCrop, CenterCrop, ElasticTransform, RGBShift, Rotate,
                            Compose, ToFloat, FromFloat, RandomRotate90, Flip, OneOf, MotionBlur, MedianBlur, Blur,
                            Transpose,
                            ShiftScaleRotate, OpticalDistortion, GridDistortion, RandomBrightnessContrast, VerticalFlip,
                            HorizontalFlip,
                            HueSaturationValue,
                            )
import shutil
sys.path.append('..')
from dataset.segmentation import *
from config import *

AVAIL_GPUS = min(1, torch.cuda.device_count())
device = "cuda" if torch.cuda.is_available() else "cpu"
#device = 'cpu'

In [3]:
num_classes = None
if num_classes == None:
    num_classes = len(os.listdir((os.path.join(data_path, "masks_classes"))))

In [4]:
model = models.segmentation.deeplabv3_resnet101(pretrained=True,
                                                progress=True)
model.classifier = DeepLabHead(2048, num_classes=num_classes+1)
model.to(device)

/davinci-1/home/morellir/miniconda3/envs/py38/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/davinci-1/home/morellir/miniconda3/envs/py38/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [5]:
checkpoint_path = "../model_results/deeplab_k_fold_multiclass/deeplabv3_resnet101/fold_1/deeplab_k_fold_multiclass_2024_03_04_11_47_13/"

In [6]:
checkpoint = torch.load(os.path.join(checkpoint_path, "model.pth"), map_location=torch.device(device))
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

_IncompatibleKeys(missing_keys=['aux_classifier.0.weight', 'aux_classifier.1.weight', 'aux_classifier.1.bias', 'aux_classifier.1.running_mean', 'aux_classifier.1.running_var', 'aux_classifier.4.weight', 'aux_classifier.4.bias'], unexpected_keys=[])

In [7]:
train=False
val=False
test=True

images_path = "../data/cropped_data/images"
data_path = Path(images_path).parent.as_posix()
normalize_imagenet = 0

if 'cfg' in checkpoint.keys():
    cfg = checkpoint['cfg']
    normalize_imagenet = cfg.dataset.normalize_imagenet
    print('from cfg normalize imagenet', normalize_imagenet)
else:
    print('not cfg', normalize_imagenet)

shuffle=1
random_seed=123
num_workers=0
batch_size = 1

train_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "train")
val_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "val")
test_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "test")

if cfg.dataset.normalize_imagenet:
    print('imagenet normalization')
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
elif cfg.dataset.automatic_normalize:
    std = torch.tensor(params["std"]).view(1, 3, 1, 1)
    mean = torch.tensor(params["mean"]).view(1, 3, 1, 1)
    std = tuple(std.squeeze().tolist())
    mean = tuple(mean.squeeze().tolist())
    print('imagenet normalization')
else:
    print('0-1 normalization')
    mean = (0.0, 0.0, 0.0)
    std = (1.0, 1.0, 1.0)

transform = None

#transform=None
if train:
    dataset = KFoldDataframeMulticlass(data_path=data_path, df_path=train_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped, n_classes = num_classes)
elif val:
    dataset = KFoldDataframeMulticlass(data_path=data_path, df_path=val_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped, n_classes = num_classes)
elif test:
    dataset = KFoldDataframeMulticlass(data_path=data_path, df_path=test_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped, n_classes = num_classes)
    
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                            drop_last=True)

from cfg normalize imagenet 0
0-1 normalization


In [8]:
results_path = os.path.join(checkpoint_path, "results")
if not os.path.exists(results_path):
    os.makedirs(results_path)
else:
    shutil.rmtree(results_path)
    os.makedirs(results_path)

In [9]:
x, y, ym = next(iter(dataloader))

In [10]:
palette = [0, 150, 255]

In [16]:
save=False
threshold_value = 0.3

count = 0
model.eval()

with torch.no_grad():
    for i, (im, mask_batch, mask_multi_batch) in enumerate(dataloader):
        
        results = model(im.to(device))
        
        
        for ix in range(results['out'].shape[0]):
            # Create subplots with one row and three columns
            mask = mask_batch[ix]
            #pred = results['out'][ix].softmax(0)
            
            pred = pred.permute(1,2,0).detach().cpu().numpy()
            #print(np.unique(pred))
            #pred = np.where(pred > threshold_value, pred, 0)

            pred = np.argmax(pred, 2)

            pred[pred == 0] = palette[0]
            pred[pred == 1] = palette[1]
            pred[pred == 2] = palette[2]

            mask[mask == 0] = palette[0]
            mask[mask == 1] = palette[1]
            mask[mask == 2] = palette[2]

            print(np.unique(mask))
            
             
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            # Plot on the first subplot
            axes[0].imshow(im[ix].permute(1,2,0))
            axes[0].set_title('Plot 1')

            # Plot on the second subplot
            axes[1].imshow(mask.permute(1,2,0))
            axes[1].set_title('Plot 2')

            # Plot on the third subplot
            axes[2].imshow(pred)
            axes[2].set_title('Plot 3')

            # Adjust layout to prevent clipping of titles
            plt.tight_layout()

            # Show the plots
            plt.show()
            
            
            

            name = dataset.images_file_names[count]
            if save:
                #print(name)
                cv2.imwrite(os.path.join(results_path, f"{name}"), np.squeeze(thresholded_image*255))
                #plt.imsave(os.path.join(results_path, f"{name}"),np.squeeze(thresholded_image), cmap='gray')
                
            count += 1

AttributeError: 'numpy.ndarray' object has no attribute 'permute'

In [ ]:
results_path = os.path.join(checkpoint, "heatmap_results")
if not os.path.exists(results_path):
    os.makedirs(results_path)
else:
    shutil.rmtree(results_path)
    os.makedirs(results_path)

In [ ]:
import gc

count = 0
model.eval()


with torch.no_grad():
    for i, (im, mask) in enumerate(dataloader):
        print(i)
        results = model(im.to(device))
        
        
        for ix in range(results['out'].shape[0]):
            # Create subplots with one row and three columns

            pred = results['out'][ix].sigmoid().permute(1,2,0).detach().cpu().numpy()
            name = dataset.images_file_names[count]

            plt.imsave(os.path.join(results_path, f"{name}"),np.squeeze(pred), cmap='gray')
            count += 1